Code to set path root

In [1]:
import sys
import os
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


sys.path.append(os.path.abspath(".."))

# Training model on `fight-weaponized-other-dataset` with 64x64 Image Sizes
* using `datasets`, `transforms` module from `torchvison`
* using `dataloader` module from `torch.utils.data`

## Importing necessary Modules

In [2]:
# Import torch libraries
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn

# Import modules
from modules.architectures.Architecture import Architecture, ResidualBlock
from modules.helper.Trainer import Trainer
from modules.helper.Plotter import plot_training_metrics, plot_testing_history
from modules.helper.Tester import  Tester

Check if CUDA is used

In [3]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device name:", torch.cuda.get_device_name(0))
    print("Current device index:", torch.cuda.current_device())
    print("Device count:", torch.cuda.device_count())
else:
    print("Running on CPU")

CUDA available: True
CUDA device name: NVIDIA GeForce RTX 4070 Laptop GPU
Current device index: 0
Device count: 1


### Use datasets, dataloader and transforms for loading training Dataset

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])
train_dataset = datasets.ImageFolder(
    root = "../datasets/fight-weaponized-other-dataset/train",
    transform = train_transform
)

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    shuffle=True
)

print("Total Batches => ", len(train_dataloader))

Total Batches =>  67


### Use datasets, dataloader and transforms for loading validation Dataset

In [5]:
val_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

val_dataset = datasets.ImageFolder(
    root = "../datasets/fight-weaponized-other-dataset/val",
    transform = val_transform
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=32,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("Total Batches => ", len(val_dataloader))

Total Batches =>  15


### Using Model Architecture:
* 10 Convolutional Layers
    - Conv2D
    - BatchNorm2D
    - ReLu
    - MaxPool2D (Optional)
* 1 Linear Layer
* SDG Optimizer

In [6]:
model = Architecture().to("cuda")

### Adding 8 blocks (MaxPool2D in each second block)

In [ ]:
in_channels = 3
out_channels = 8
size = 64

model_blocks = []

for i in range(1, 9):
    conv = nn.Conv2d(in_channels, out_channels, 3, 1, 1)
    batch_norm = nn.BatchNorm2d(out_channels)
    model_blocks.extend(
        [conv, batch_norm, nn.ReLU()]
    )
    if i%2==0:
        model_blocks.append(nn.MaxPool2d(2,2))
        size = size//2
    
    in_channels = out_channels
    out_channels = out_channels * 2

print(f"Final In Channels = {in_channels}")
print(f"Final Out Channels = {out_channels}")
print(f"Final Shape = {size}")

Final In Channels = 1024
Final Out Channels = 2048
Final Shape = 4


In [8]:
model = model.add(
    # Conv Blocks
    *model_blocks,
    
    # Flatten
    nn.Flatten(),

    nn.Linear(out_channels * size * size, out_channels),
    nn.ReLU(),
    nn.Linear(out_channels, 128),
    nn.ReLU(),
    nn.Linear(128, 3)
    
    )

### Use Trainer to train and check validations

In [9]:
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [10]:
trainer = Trainer(
    model, 
    train_dataloader, 
    val_dataloader, 
    optimizer=optimizer, 
    num_classes=3,
    criterion=criterion,
    device="cuda",
    save_dir="../models/experiment3/",
    save_checkpoints=1,
    print_every=10
    )

In [11]:
history = trainer.fit(100)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x16384 and 32768x2048)

In [ ]:
df = plot_training_metrics(history)

In [ ]:
filtered_df = df[
    (df["val_accuracy"] >= 0.8) &
    (df["val_f1"] >= 0.8)
].to_dict()

high_accuracy_df = plot_training_metrics(filtered_df)

In [ ]:
df

### Training/Validation Trend (100 epochs)
The model shows strong learning progress during training, with training loss continuously decreasing from 0.9918 to 0.0121 and training accuracy improving from 52.36% to 100%, indicating that the model successfully learned the training patterns. Validation performance improved steadily in the early epochs, with validation accuracy increasing from 59.29% to a peak of around 82.52%. However, after approximately epoch 27, the training metrics continued improving while validation metrics fluctuated, suggesting the beginning of overfitting. The best generalization performance was achieved around the mid-training stage, where validation accuracy, precision, recall, and F1-score were highest before the model started memorizing the training data.

<b>Epoch 44</b>

<b>Loss</b>
* Train Loss = 0.059317
* Valid Loss = 0.590012

<b>Training Metrics</b>
* Train Accuracy = 0.996692
* Train Precison = 0.996653
* Train Recall = 0.996763
* Train F1 = 0.996704

<b>Validation Accuracy</b>
* Validation Accuracy = 0.825221
* Validation Precision = 0.828700
* Validation Recall = 0.824604
* Validation F1 = 0.825825

Here, the `Epoch 44` is selected dispite `Epoch 78` has the higher val metrics. It's because the Epoch 78 has `1.00` training Metrics which represents a serious overfitting.

The model still has a serious overfitting though the additional ConvLayer and decreased LR helped stabalize the curves. 

## Use Tester Module to Test Model

Load Model with State Dict

In [ ]:
import copy

test_scores = []

# Transforms of Data
test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# Dataset Loading From Image dir
test_dataset = datasets.ImageFolder(
    root="../datasets/fight-weaponized-other-dataset/test", 
    transform = test_transform 
    )

# DataLoader
test_loader = DataLoader(
    dataset=test_dataset, 
    batch_size=64,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
    )

# Loops for testing
for i in range(100):
    test_model = copy.deepcopy(model)

    checkpoint = torch.load(
        f"../models/experiment3/model_epoch_{i+1}.pt",
        map_location="cuda"
    )

    test_model.load_state_dict(checkpoint["model"])

    tester = Tester(
        test_model,
        test_loader,
        3,
        torch.nn.CrossEntropyLoss(),
        "cuda"
    )

    result = tester.test(return_predictions=True)

    test_scores.append(result)


In [ ]:
test_metrics_df = plot_testing_history(test_scores)

In [ ]:
test_metrics_df

### Test Performance Trend (100 epochs)
The test metrics show a clear learning curve followed by saturation: performance improves rapidly in the early epochs as loss drops and accuracy rises from about 0.62 to ~0.76, then continues to improve more gradually until reaching a stable plateau around 0.81–0.83 accuracy with closely aligned precision, recall, and F1 scores. After roughly epochs 25–30, gains become marginal and the model mainly oscillates within a narrow performance band, indicating it has reached its generalization limit on the dataset. In later epochs (around 70+), small fluctuations appear with occasional dips, suggesting mild overfitting and optimization noise, but no sustained improvement beyond the peak region, which occurs around epochs 71–74 with the best F1/accuracy balance near ~0.83.

<b>Epoch 74</b>

* Loss = 0.710831
* Accuracy = 0.828947
* Precision = 0.831552
* Recall = 0.830613
* F1-Score = 0.830150